# BEV visualization: predictions vs. ground truth

Top-down (ego-frame) views of one scene. Ground truth is **blue solid**, predictions are **orange dashed**, and a
line from each box centre marks its heading. The last section animates tracks over the scene.

Set `GT` / `PRED` to real files once `scripts/gpu_pipeline.sh` has run. By default this notebook falls back to the
synthetic data from `python -m src.synthetic`, which exercises the code but is **not a result**.

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))
import numpy as np
import matplotlib.pyplot as plt
from src.dataio import load_gt, load_predictions
from src.viz import plot_bev

GT, PRED, TRACKS = '../data/processed/gt_clean.json', '../results/preds_finetuned.json', '../results/tracks.json'
if not os.path.exists(PRED):
    print('Real predictions not found -> using synthetic data (NOT results)')
    if not os.path.exists('../data/synthetic/gt.json'):
        from src.synthetic import main as make_synth
        make_synth(['--out-dir', '../data/synthetic'])
    GT, PRED, TRACKS = '../data/synthetic/gt.json', '../data/synthetic/preds.json', None

samples = load_gt(GT)
preds = load_predictions(PRED, samples)
scenes = sorted({s.scene_name for s in samples.values()})
print(len(samples), 'samples in scenes', scenes)

## A scene at a glance: 6 frames, predictions with score ≥ 0.3

In [ ]:
SCENE = scenes[0]
seq = [s for s in samples.values() if s.scene_name == SCENE]
picks = np.linspace(0, len(seq) - 1, 6).astype(int)
fig, axes = plt.subplots(2, 3, figsize=(15, 10), layout='constrained')
for ax, i in zip(axes.flat, picks):
    s = seq[i]
    plot_bev(s.gt, preds[s.token], ax=ax, extent=50, score_threshold=0.3,
             title=f"{SCENE} frame {i} ({s.condition.get('lighting', '?')}, {s.condition.get('weather', '?')})")
plt.show()

## Detection quality by distance, per class

One panel per class, so a class is never identified by color alone. Buckets with no ground truth are left empty
rather than drawn as zero. Note that the devkit class ranges cap pedestrians and cyclists at 40 m and barriers at 30 m.

In [ ]:
from src.eval.slice_eval import run, bucket_names
from src.classes import CLASSES
results, *_ = run(samples, preds)
buckets = bucket_names()
SLOT = ['#2a78d6', '#eb6834', '#1baf7a', '#eda100', '#e87ba4']  # categorical slots 1-5, fixed order = CLASSES order

fig, axes = plt.subplots(1, len(CLASSES), figsize=(16, 3.4), sharey=True, layout='constrained')
for ax, cls, color in zip(axes, CLASSES, SLOT):
    aps = [results[('distance', b)].per_class[cls].ap for b in buckets]
    ngt = [results[('distance', b)].per_class[cls].n_gt for b in buckets]
    for x, (ap, n) in enumerate(zip(aps, ngt)):
        if n == 0:
            ax.text(x, 0.04, 'no GT', ha='center', fontsize=8, color='0.45')
            continue
        ax.bar(x, ap, width=0.6, color=color)
        ax.text(x, ap + 0.02, f'{ap:.2f}\nn={n}', ha='center', va='bottom', fontsize=8, color='0.2')
    ax.set_xticks(range(len(buckets)), buckets)
    ax.set_title(cls)
    ax.set_ylim(0, 1.05)
    ax.grid(axis='y', color='0.9'); ax.set_axisbelow(True)
    for side in ('top', 'right'):
        ax.spines[side].set_visible(False)
axes[0].set_ylabel('AP (nuScenes, mean over 0.5/1/2/4 m)')
plt.show()

# Table view of the same numbers
print(f"{'class':<12}" + ''.join(f'{b:>16}' for b in buckets))
for cls in CLASSES:
    cells = [results[('distance', b)].per_class[cls] for b in buckets]
    print(f'{cls:<12}' + ''.join(f"{(f'{c.ap:.3f}' if c.n_gt else '-'):>9} (n={c.n_gt:>3})" for c in cells))

## Tracks over time (GIF)

Each track keeps one color for its lifetime; GT boxes are drawn thin gray. An ID switch shows up as a box changing color mid-sequence.

In [ ]:
from matplotlib.animation import FuncAnimation, PillowWriter
from src.track import track_all
from src.boxes import bev_corners

tracks = track_all(samples, preds)
cmap = plt.get_cmap('tab20')
color_of = {}
def track_color(tid):
    return color_of.setdefault(tid, cmap(len(color_of) % 20))

fig, ax = plt.subplots(figsize=(7, 7))
def draw(i):
    ax.clear()
    s = seq[i]
    for g in s.gt:
        c = bev_corners(g.translation, g.size[:2], g.yaw)
        ax.fill(c[:, 0], c[:, 1], fill=False, edgecolor='0.6', lw=0.8)
    for t in tracks[s.token]:
        c = bev_corners(t.translation, t.size[:2], t.yaw)
        ax.fill(c[:, 0], c[:, 1], fill=False, edgecolor=track_color(t.instance_id), lw=2)
        ax.text(t.translation[0] + 1, t.translation[1] + 1, t.instance_id.split('/')[-1], fontsize=6, color='0.2', clip_on=True)
    ax.plot(0, 0, marker=(3, 0, -90), color='k', ms=10)
    ax.set_xlim(-50, 50); ax.set_ylim(-50, 50); ax.set_aspect('equal')
    ax.set_title(f'{SCENE}  frame {i}/{len(seq) - 1}  (gray = GT, colored = tracks, label = track id)', fontsize=9)
anim = FuncAnimation(fig, draw, frames=len(seq))
os.makedirs('../results', exist_ok=True)
out = '../results/bev_tracks.gif' if TRACKS else '../results/bev_tracks_SYNTHETIC.gif'
anim.save(out, writer=PillowWriter(fps=2))
plt.close(fig)
print('saved', out)